# Absorbing Failure States — Visual Tests
Verify that the absorbing region works correctly in the FourRooms environment.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from point_env import PointEnv

## Helper: Plot FourRooms with trajectory

In [ ]:
def plot_trajectory(env, trajectory, title='', region_bounds=None):
    """Plot the maze, trajectory, absorbing region, start/goal/absorb points."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    # Draw walls
    walls = env.walls
    for i in range(walls.shape[0]):
        for j in range(walls.shape[1]):
            if walls[i, j] == 1:
                ax.add_patch(patches.Rectangle((j, i), 1, 1, 
                    facecolor='black', edgecolor='gray', linewidth=0.5))
    
    # Draw absorbing region
    if region_bounds is not None:
        lower, upper = region_bounds
        width = upper[0] - lower[0]
        height = upper[1] - lower[1]
        ax.add_patch(patches.Rectangle(
            (lower[1], lower[0]), height, width,
            facecolor='red', alpha=0.15, edgecolor='red', 
            linewidth=2, linestyle='--', label='Absorbing Region'))
    
    # Extract trajectory data
    states = np.array([t['state'] for t in trajectory])
    absorbed_flags = [t['absorbed'] for t in trajectory]
    
    # Find absorption point
    absorb_idx = None
    for i, a in enumerate(absorbed_flags):
        if a:
            absorb_idx = i
            break
    
    # Plot trajectory before absorption (blue)
    if absorb_idx is not None:
        pre = states[:absorb_idx+1]
        post = states[absorb_idx:]
        ax.plot(pre[:, 1], pre[:, 0], 'b-', linewidth=1.5, alpha=0.7, label='Before absorption')
        ax.scatter(post[0, 1], post[0, 0], c='red', s=200, marker='X', 
                   zorder=5, label=f'Absorbed (step {absorb_idx})')
    else:
        ax.plot(states[:, 1], states[:, 0], 'b-', linewidth=1.5, alpha=0.7, label='Trajectory')
    
    # Start and goal
    ax.scatter(states[0, 1], states[0, 0], c='green', s=150, marker='o', zorder=5, label='Start')
    ax.scatter(env.goal[1], env.goal[0], c='gold', s=200, marker='*', zorder=5, label='Goal')
    
    ax.set_xlim(-0.5, walls.shape[1] + 0.5)
    ax.set_ylim(-0.5, walls.shape[0] + 0.5)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('y')
    ax.set_ylabel('x')
    ax.invert_yaxis()
    plt.tight_layout()
    return fig, ax

In [ ]:
def run_episode(env, actions, max_steps=200):
    """Run an episode collecting trajectory data."""
    trajectory = []
    obs = env.reset()
    trajectory.append({
        'state': env.state.copy(),
        'reward': 0.0,
        'absorbed': False,
    })
    
    for i in range(max_steps):
        if callable(actions):
            action = actions(i, env.state)
        else:
            action = actions[i % len(actions)]
        obs, rew, done, info = env.step(action)
        trajectory.append({
            'state': env.state.copy(),
            'reward': rew,
            'absorbed': info.get('absorbed', False),
        })
    return trajectory

## Test 1: Agent enters absorbing region and freezes

In [ ]:
region_bounds = ([0, 5], [5, 11])

# Start agent in bottom-left room, goal in bottom-right
# Agent will move up-right through the doorway into the absorbing region
env = PointEnv(
    walls='FourRooms',
    region_bounds=region_bounds,
    fixed_start_end=[np.array([2, 4], dtype=float), np.array([10, 8], dtype=float)]
)

# Use random actions so the agent explores and eventually enters the region
np.random.seed(42)
trajectory = run_episode(env, 
    lambda i, s: np.random.uniform(-1, 1, size=2).astype(np.float32), 
    max_steps=300)

# Check absorption happened
absorbed_steps = [i for i, t in enumerate(trajectory) if t['absorbed']]
if absorbed_steps:
    print(f'Absorbed at step: {absorbed_steps[0]}')
    print(f'Reward after absorption: {[t["reward"] for t in trajectory[absorbed_steps[0]:absorbed_steps[0]+5]]}')
    
    # Verify state is frozen
    frozen_states = [trajectory[i]['state'] for i in range(absorbed_steps[0], min(absorbed_steps[0]+10, len(trajectory)))]
    all_same = all(np.allclose(s, frozen_states[0]) for s in frozen_states)
    print(f'State frozen after absorption: {all_same}')
else:
    print('Agent did not enter absorbing region — try rerunning or increasing max_steps')

fig, ax = plot_trajectory(env, trajectory, 
    title='Test 1: Agent enters absorbing region and freezes',
    region_bounds=region_bounds)
plt.show()

## Test 2: Reset clears absorption

In [ ]:
# First episode: get absorbed (start near the region boundary)
env = PointEnv(
    walls='FourRooms',
    region_bounds=region_bounds,
    fixed_start_end=[np.array([2, 4], dtype=float), np.array([10, 8], dtype=float)]
)
np.random.seed(42)
traj1 = run_episode(env, lambda i, s: np.random.uniform(-1, 1, size=2).astype(np.float32), max_steps=300)
print(f'Episode 1 - Absorbed: {env._is_absorbed}')

# Reset and run second episode going down-left (away from region)
traj2 = run_episode(env, lambda i, s: np.array([0.8, -0.8], dtype=np.float32), max_steps=100)
absorbed_in_traj2 = any(t['absorbed'] for t in traj2)
print(f'Episode 2 - Absorbed after reset: {absorbed_in_traj2}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for ax, traj, title in [(ax1, traj1, 'Episode 1: Gets absorbed'), 
                          (ax2, traj2, 'Episode 2: After reset')]:
    walls = env.walls
    for i in range(walls.shape[0]):
        for j in range(walls.shape[1]):
            if walls[i, j] == 1:
                ax.add_patch(patches.Rectangle((j, i), 1, 1, facecolor='black', edgecolor='gray', linewidth=0.5))
    lower, upper = region_bounds
    ax.add_patch(patches.Rectangle((lower[1], lower[0]), upper[1]-lower[1], upper[0]-lower[0],
        facecolor='red', alpha=0.15, edgecolor='red', linewidth=2, linestyle='--'))
    states = np.array([t['state'] for t in traj])
    absorb_idx = next((i for i, t in enumerate(traj) if t['absorbed']), None)
    if absorb_idx is not None:
        ax.plot(states[:absorb_idx+1, 1], states[:absorb_idx+1, 0], 'b-', linewidth=1.5, alpha=0.7)
        ax.scatter(states[absorb_idx, 1], states[absorb_idx, 0], c='red', s=200, marker='X', zorder=5)
    else:
        ax.plot(states[:, 1], states[:, 0], 'b-', linewidth=1.5, alpha=0.7)
    ax.scatter(states[0, 1], states[0, 0], c='green', s=150, marker='o', zorder=5)
    ax.scatter(env.goal[1], env.goal[0], c='gold', s=200, marker='*', zorder=5)
    ax.set_xlim(-0.5, walls.shape[1]+0.5)
    ax.set_ylim(-0.5, walls.shape[0]+0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_title(title, fontsize=13)
plt.tight_layout()
plt.show()

## Test 3: No absorbing region — agent moves freely everywhere

In [ ]:
env_no_region = PointEnv(
    walls='FourRooms',
    region_bounds=None,
    fixed_start_end=[np.array([0, 0], dtype=float), np.array([10, 8], dtype=float)]
)

traj_free = run_episode(env_no_region, lambda i, s: np.array([0.5, 0.8], dtype=np.float32), max_steps=150)

absorbed_any = any(t['absorbed'] for t in traj_free)
print(f'Any absorption without region_bounds: {absorbed_any}')

fig, ax = plot_trajectory(env_no_region, traj_free,
    title='Test 3: No absorbing region — agent moves freely',
    region_bounds=None)

# Show where the region *would* be for reference
ax.add_patch(patches.Rectangle((5, 0), 6, 5,
    facecolor='none', edgecolor='gray', linewidth=1, linestyle=':', label='(Would-be region)'))
ax.legend(loc='upper right', fontsize=9)
plt.show()

## Test 4: Reward timeline — shows reward drops to 0 at absorption

In [ ]:
env = PointEnv(
    walls='FourRooms',
    region_bounds=region_bounds,
    fixed_start_end=[np.array([2, 4], dtype=float), np.array([10, 8], dtype=float)]
)

np.random.seed(42)
trajectory = run_episode(env, 
    lambda i, s: np.random.uniform(-1, 1, size=2).astype(np.float32), 
    max_steps=300)

rewards = [t['reward'] for t in trajectory]
absorbed = [t['absorbed'] for t in trajectory]
absorb_step = next((i for i, a in enumerate(absorbed) if a), None)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(rewards, 'b-', linewidth=1)
if absorb_step:
    ax1.axvline(x=absorb_step, color='red', linestyle='--', label=f'Absorbed (step {absorb_step})')
    ax1.legend()
ax1.set_ylabel('Reward')
ax1.set_title('Reward and absorption over time')

ax2.fill_between(range(len(absorbed)), [int(a) for a in absorbed], alpha=0.3, color='red')
ax2.set_ylabel('Absorbed')
ax2.set_xlabel('Step')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()